## 2.a Setup, path, dan data mentah

1. Membangun `PROJECT_ROOT` dengan dependensi yang dibutuhkan: `polars`, `duckdb`, `geopandas`, `pyarrow`

In [1]:
from pathlib import Path

import duckdb
import polars as pl

pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(20)

PROJECT_ROOT = (
    Path("/workspace")
    if Path("/workspace/data").exists()
    else Path.cwd().resolve().parent
)
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "fire_archive_SV-C2_812847.csv"
BOUNDARY_PATH = PROJECT_ROOT / "data" / "external" / "indonesia_provinces.geojson"
AREA_PATH = PROJECT_ROOT / "data" / "external" / "luas_provinsi.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "hotspot_provinsi_bulanan.parquet"

for path in (RAW_PATH, BOUNDARY_PATH):
    if not path.exists():
        raise FileNotFoundError(f"File tidak ditemukan: {path}")

N_RAW_MILESTONE1 = 1_535_566 

lf = pl.scan_csv(RAW_PATH)
n_raw = lf.select(pl.len()).collect().item()
print(f"Baris mentah: {n_raw:,}")
if n_raw != N_RAW_MILESTONE1:
    print(f"PERINGATAN: berbeda dari Milestone 1 ({N_RAW_MILESTONE1:,}). Cek file sumber.")

audit = [("mentah", n_raw)]

Baris mentah: 1,535,566


## PENJELASAN 
1. Pada cell ini fokus pada import library, lalu mengatur struktur file dari hasil notebook data profiling sebelumnya.
2. Data dibaca dengan `scan_csv` (lazy) sehingga file tidak dimuat penuh ke memori. Polars baru memprosesnya saat `.collect()` dipanggil.
3. Jumlah baris mentah dibandingkan dengan hasil Milestone 1 (1.535.566 baris) sebagai pengecekan bahwa file sumber sama.

## 2.b Profiling dengan DuckDB `SUMMARIZE`

Ringkasan tipe data, min/max, jumlah nilai unik, dan persentase null seluruh kolom dalam satu perintah.

In [2]:
con = duckdb.connect()
profil = con.execute(
    f"SUMMARIZE SELECT * FROM read_csv_auto('{RAW_PATH.as_posix()}')"
).pl()
profil.select(["column_name", "column_type", "min", "max", "approx_unique", "null_percentage"])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,null_percentage
str,str,str,str,i64,"decimal[9,2]"
"""latitude""","""DOUBLE""","""-10.97676""","""5.87273""",840522,0.00
"""longitude""","""DOUBLE""","""95.01992""","""140.9774""",1126093,0.00
"""brightness""","""DOUBLE""","""208.0""","""367.0""",6175,0.00
"""scan""","""DOUBLE""","""0.32""","""0.8""",47,0.00
"""track""","""DOUBLE""","""0.36""","""0.78""",43,0.00
"""acq_date""","""DATE""","""2016-01-01""","""2026-06-30""",3918,0.00
"""acq_time""","""VARCHAR""","""0308""","""2004""",525,0.00
"""satellite""","""VARCHAR""","""SNPP""","""SNPP""",1,0.00
"""instrument""","""VARCHAR""","""SNPP""","""SNPP""",1,0.00


## PENJELASAN ##
1. `SUMMARIZE` memberi gambaran awal kualitas data: tipe tiap kolom, rentang nilai, dan kolom yang punya null.
2. Hasil ini dipakai untuk memutuskan kolom mana yang wajib terisi dan batas nilai yang masuk akal di tahap validasi.

## 2.c Missing values

In [3]:
null_summary = (
    lf.select(pl.all().null_count())
    .collect()
    .transpose(include_header=True, header_name="kolom", column_names=["jumlah_null"])
    .with_columns((pl.col("jumlah_null") / n_raw * 100).round(4).alias("persen_null"))
    .sort("jumlah_null", descending=True)
)
print(f"Kolom dengan null: {null_summary.filter(pl.col('jumlah_null') > 0).height} dari {null_summary.height}")
null_summary

Kolom dengan null: 0 dari 15


kolom,jumlah_null,persen_null
str,u32,f64
"""latitude""",0,0.0
"""longitude""",0,0.0
"""brightness""",0,0.0
"""scan""",0,0.0
"""track""",0,0.0
"""acq_date""",0,0.0
"""acq_time""",0,0.0
"""satellite""",0,0.0
"""instrument""",0,0.0


In [4]:
KOLOM_WAJIB = ["latitude", "longitude", "acq_date", "acq_time", "confidence", "type"]

lf_nonnull = lf.drop_nulls(subset=KOLOM_WAJIB)
n_nonnull = lf_nonnull.select(pl.len()).collect().item()
print(f"Baris dibuang karena null di kolom wajib: {n_raw - n_nonnull:,}")
audit.append(("setelah drop null kolom wajib", n_nonnull))

Baris dibuang karena null di kolom wajib: 0


## PENJELASAN ##
1. Null dihitung untuk seluruh kolom supaya terlihat bahwa kolom lain memang bersih.
2. Tidak ada imputasi (median/modus) karena kolom wajib berisi posisi, waktu, dan label karena jika mengisi nilai buatan pada kolom-kolom ini akan menciptakan titik api fiktif.
3. Langkah ini juga mencegah filter `confidence != "l"` membuang baris null secara diam-diam.

## 2.d Validasi dan penanganan duplikasi

In [5]:
KUNCI = ["latitude", "longitude", "acq_date", "acq_time"]

dup_check = (
    lf_nonnull.group_by(KUNCI)
    .agg(pl.len().alias("jumlah"))
    .filter(pl.col("jumlah") > 1)
    .collect()
)
print(f"Grup kunci yang duplikat: {dup_check.height:,}")

lf_dedup = lf_nonnull.unique(subset=KUNCI, keep="first", maintain_order=True)
n_dedup = lf_dedup.select(pl.len()).collect().item()
print(f"Baris dibuang karena duplikat: {n_nonnull - n_dedup:,}")
audit.append(("setelah deduplikasi", n_dedup))
dup_check.head(5)

Grup kunci yang duplikat: 0
Baris dibuang karena duplikat: 0


latitude,longitude,acq_date,acq_time,jumlah
f64,f64,str,i64,u32


## PENJELASAN ##
1. Satu titik hotspot dianggap unik berdasarkan kombinasi latitude, longitude, tanggal, dan jam akuisisi 
2. Data dikelompokkan berdasarkan kombinasi itu lalu dicari yang muncul lebih dari satu kali.

## 2.e Filtering domain

In [6]:
print(lf_dedup.group_by("confidence").agg(pl.len().alias("jumlah")).sort("jumlah", descending=True).collect())
print(lf_dedup.group_by("type").agg(pl.len().alias("jumlah")).sort("type").collect())

shape: (3, 2)
┌────────────┬─────────┐
│ confidence ┆ jumlah  │
│ ---        ┆ ---     │
│ str        ┆ u32     │
╞════════════╪═════════╡
│ n          ┆ 1386873 │
│ l          ┆ 101396  │
│ h          ┆ 47297   │
└────────────┴─────────┘
shape: (4, 2)
┌──────┬─────────┐
│ type ┆ jumlah  │
│ ---  ┆ ---     │
│ i64  ┆ u32     │
╞══════╪═════════╡
│ 0    ┆ 1416095 │
│ 1    ┆ 32173   │
│ 2    ┆ 79308   │
│ 3    ┆ 7990    │
└──────┴─────────┘


In [7]:
FILTER_EXCLUDE_LOW_CONFIDENCE = True
FILTER_ONLY_VEGETATION_FIRES = True

lf_filtered = lf_dedup
if FILTER_EXCLUDE_LOW_CONFIDENCE:
    lf_filtered = lf_filtered.filter(pl.col("confidence") != "l")
if FILTER_ONLY_VEGETATION_FIRES:
    lf_filtered = lf_filtered.filter(pl.col("type") == 0)

n_filtered = lf_filtered.select(pl.len()).collect().item()
print(f"Baris sebelum filter : {n_dedup:,}")
print(f"Baris setelah filter : {n_filtered:,}")
print(f"Persentase dibuang   : {(1 - n_filtered / n_dedup) * 100:.2f}%")
audit.append(("setelah filter confidence & type", n_filtered))

Baris sebelum filter : 1,535,566
Baris setelah filter : 1,317,362
Persentase dibuang   : 14.21%


## PENJELASAN ##
1. Dua tabel di atas menampilkan nilai kategori yang benar-benar ada di data, supaya filter ditulis berdasarkan nilai nyata, bukan asumsi.
2. Filter `confidence != 'l'` membuang deteksi berkepercayaan rendah yang rawan false positive.
3. Filter `type == 0` mempertahankan titik yang terklasifikasi sebagai kebakaran vegetasi, dan membuang gunung berapi, sumber panas statis (industri), dan offshore, karena fokus analisis adalah kebakaran hutan/lahan.
4. Hasil terakhir setelah dilakukan filter sebanyak 1,317,362 baris.

## 2.f Deteksi outlier

In [8]:
NUM_COLS = ["brightness", "bright_t31", "frp", "scan", "track"]

q = lf_filtered.select(
    [pl.col(c).min().alias(f"{c}|min") for c in NUM_COLS]
    + [pl.col(c).quantile(0.25).alias(f"{c}|q1") for c in NUM_COLS]
    + [pl.col(c).quantile(0.75).alias(f"{c}|q3") for c in NUM_COLS]
    + [pl.col(c).quantile(0.99).alias(f"{c}|p99") for c in NUM_COLS]
    + [pl.col(c).max().alias(f"{c}|max") for c in NUM_COLS]
).collect().row(0, named=True)

batas = {}
for c in NUM_COLS:
    iqr = q[f"{c}|q3"] - q[f"{c}|q1"]
    batas[c] = (q[f"{c}|q1"] - 1.5 * iqr, q[f"{c}|q3"] + 1.5 * iqr)

n_out = lf_filtered.select(
    [(pl.col(c) < batas[c][0]).sum().alias(f"{c}|bawah") for c in NUM_COLS]
    + [(pl.col(c) > batas[c][1]).sum().alias(f"{c}|atas") for c in NUM_COLS]
).collect().row(0, named=True)

ringkasan_outlier = pl.DataFrame([
    {
        "kolom": c,
        "min": q[f"{c}|min"],
        "p99": q[f"{c}|p99"],
        "max": q[f"{c}|max"],
        "batas_bawah_iqr": round(batas[c][0], 2),
        "batas_atas_iqr": round(batas[c][1], 2),
        "outlier_bawah": n_out[f"{c}|bawah"],
        "outlier_atas": n_out[f"{c}|atas"],
        "persen_outlier": round((n_out[f"{c}|bawah"] + n_out[f"{c}|atas"]) / n_filtered * 100, 2),
    }
    for c in NUM_COLS
])
ringkasan_outlier

kolom,min,p99,max,batas_bawah_iqr,batas_atas_iqr,outlier_bawah,outlier_atas,persen_outlier
str,f64,f64,f64,f64,f64,i64,i64,f64
"""brightness""",295.0,367.0,367.0,307.57,362.85,157266,45471,15.39
"""bright_t31""",150.0,311.33,378.18,276.31,313.19,39160,7083,3.51
"""frp""",0.0,54.83,1322.99,-6.11,17.17,0,113671,8.63
"""scan""",0.32,0.73,0.8,0.23,0.67,0,30070,2.28
"""track""",0.36,0.76,0.78,0.12,0.81,0,0,0.0


## PENJELASAN ##
1. Metode IQR dipakai sebagai titik awal pada lima kolom numerik dengan menghitung Batas bawah dan atas agar mengetahui sebaran data secara keseluruhan.
2. `frp` (Fire Radiative Power) miring ke kanan karena dominan titik api kecil dan hanya sedikit yang besar.
3. `brightness` adalah suhu kecerahan kanal api dalam satuan Kelvin, lalu untuk `bright_t31` adalah suhu latar belakang piksel, bukan suhu api. `scan` dan `track` merupakan ukuran piksel VIIRS yang membesar ke arah tepi sapuan satelit, sehingga nilai besar adalah efek geometri sensor.
4. Maka dari itu, outlier tidak dihapus karena dengan menghapus kebakaran besar menghilangkan kejadian nyata, dan `frp_total`/`frp_max` pada agregasi justru bergantung pada nilai ekstrem tersebut. Validitas data dijaga lewat aturan batas domain di bagian validasi (nilai negatif, koordinat di luar Indonesia, kategori tidak dikenal).

## 2.g Spatial join: titik hotspot → provinsi

In [9]:
import geopandas as gpd

PROVINCE_NAME_COLUMN = "NAME_1"
PROV = PROVINCE_NAME_COLUMN

provinces = gpd.read_file(BOUNDARY_PATH).to_crs("EPSG:4326")
print(provinces.columns.tolist())

assert PROV in provinces.columns, f"Kolom {PROV} tidak ada di file batas"
assert provinces[PROV].is_unique, (
    f"Nama provinsi di {PROV} tidak unik; gabungkan dulu dengan provinces.dissolve(by='{PROV}')"
)
print(f"Jumlah provinsi pada file batas: {len(provinces)}")

['GID_1', 'GID_0', 'COUNTRY', 'NAME_1', 'VARNAME_1', 'NL_NAME_1', 'TYPE_1', 'ENGTYPE_1', 'CC_1', 'HASC_1', 'ISO_1', 'geometry']
Jumlah provinsi pada file batas: 34


In [10]:
from shapely.validation import explain_validity, make_valid

invalid_mask = ~provinces.geometry.is_valid
print(f"Polygon tidak valid: {int(invalid_mask.sum())}")

luas_sebelum = provinces.to_crs("EPSG:6933").geometry.area / 1_000_000

if invalid_mask.any():
    print(
        provinces.loc[invalid_mask, [PROV]]
        .assign(alasan=provinces.loc[invalid_mask, "geometry"].apply(explain_validity))
        .to_string()
    )
    provinces["geometry"] = provinces.geometry.apply(
        lambda g: g if g.is_valid else make_valid(g)
    )

assert provinces.geometry.is_valid.all(), "Masih ada polygon tidak valid setelah make_valid"

luas_sesudah = provinces.to_crs("EPSG:6933").geometry.area / 1_000_000
perubahan_luas = (
    pl.DataFrame({
        PROV: provinces[PROV].tolist(),
        "luas_sebelum_km2": luas_sebelum.tolist(),
        "luas_sesudah_km2": luas_sesudah.tolist(),
    })
    .with_columns((pl.col("luas_sesudah_km2") - pl.col("luas_sebelum_km2")).alias("selisih_km2"))
    .filter(pl.col("selisih_km2").abs() > 1e-6)
)
print(f"Provinsi dengan luas berubah setelah perbaikan: {perubahan_luas.height}")
perubahan_luas

Polygon tidak valid: 2
             NAME_1                                                 alasan
22            Papua                     Self-intersection[138.8514 -8.342]
26  SulawesiSelatan  Too few points in geometry component[121.203 -6.7779]
Provinsi dengan luas berubah setelah perbaikan: 0


NAME_1,luas_sebelum_km2,luas_sesudah_km2,selisih_km2
str,f64,f64,f64


## PENJELASAN ##
1. Pada pengecekan awal ditemukan 2 polygon provinsi yang tidak valid secara geometri membuat operasi spasial (spatial join dan perhitungan luas) memberi hasil yang tidak teratur.
2. Sel di atas menampilkan provinsi mana yang bermasalah beserta alasannya, lalu memperbaikinya dengan `make_valid` sebelum spatial join dijalankan. Setelah perbaikan, semua polygon dipastikan valid lewat `assert`.
3. Tabel `perubahan_luas` menunjukkan apakah perbaikan mengubah luas provinsi. Jika selisihnya kecil, perbaikan tidak berdampak pada normalisasi per luas wilayah.

In [11]:
df_points = lf_filtered.select(
    "latitude", "longitude", "acq_date", "acq_time", "confidence", "type", "frp"
).collect().to_pandas()

points = gpd.GeoDataFrame(
    df_points,
    geometry=gpd.points_from_xy(df_points["longitude"], df_points["latitude"]),
    crs="EPSG:4326",
)
n_points = len(points)
assert n_points == n_filtered

joined = gpd.sjoin(
    points, provinces[[PROV, "geometry"]], how="left", predicate="within"
)

n_dup_join = len(joined) - n_points
print(f"Baris tambahan akibat titik cocok ke >1 polygon: {n_dup_join:,}")
if n_dup_join > 0:
    idx_ganda = joined.index[joined.index.duplicated(keep=False)].unique()
    print(joined.loc[idx_ganda, ["latitude", "longitude", PROV]].sort_index().head(10))
    joined = joined[~joined.index.duplicated(keep="first")]
assert len(joined) == n_points
audit.append(("setelah spatial join (tanpa duplikasi)", len(joined)))

n_unmatched = int(joined[PROV].isna().sum())
print(f"Titik tidak ter-assign ke provinsi: {n_unmatched:,} ({n_unmatched / len(joined) * 100:.2f}%)")

joined = joined[joined[PROV].notna()].copy()
print(f"Baris setelah drop unmatched: {len(joined):,}")
audit.append(("setelah drop titik tanpa provinsi", len(joined)))

Baris tambahan akibat titik cocok ke >1 polygon: 2
         latitude  longitude           NAME_1
621202   -4.14568  105.75489          Lampung
621202   -4.14568  105.75489  SumateraSelatan
1066127  -2.13565  103.25940            Jambi
1066127  -2.13565  103.25940  SumateraSelatan
Titik tidak ter-assign ke provinsi: 2,880 (0.22%)
Baris setelah drop unmatched: 1,314,482


## PENJELASAN ##
1. Setiap titik hotspot diubah menjadi objek geometri dari longitude/latitude, lalu dicocokkan dengan polygon provinsi (GADM, https://gadm.org/download_country.html) dengan predikat `within`. Kedua data memakai CRS yang sama (EPSG:4326).
2. Pengecekan duplikasi hasil join menemukan 2 titik yang cocok ke dua polygon sekaligus, yaitu di perbatasan Lampung dan Sumatera Selatan, serta Jambi dan Sumatera Selatan, tempat polygon GADM saling tumpang tindih. Duplikasi dibuang dengan `keep="first"`, sehingga kedua titik ditetapkan ke salah satu provinsi secara arbitrer. Dampaknya 2 dari sekitar 1,3 juta titik dan dapat diabaikan.
3. Titik yang tidak jatuh di polygon provinsi mana pun bernilai null.

## 2.h Agregasi provinsi × bulan, dinormalisasi per luas wilayah

In [12]:
prov_equal_area = provinces.to_crs("EPSG:6933")
luas_provinsi = pl.DataFrame({
    PROV: prov_equal_area[PROV].tolist(),
    "luas_km2": (prov_equal_area.geometry.area / 1_000_000).tolist(),  
})
print(luas_provinsi.sort("luas_km2", descending=True).head(10))

AREA_PATH.parent.mkdir(parents=True, exist_ok=True)
luas_provinsi.write_csv(AREA_PATH)
print(f"Tersimpan: {AREA_PATH}")

shape: (10, 2)
┌──────────────────┬───────────────┐
│ NAME_1           ┆ luas_km2      │
│ ---              ┆ ---           │
│ str              ┆ f64           │
╞══════════════════╪═══════════════╡
│ Papua            ┆ 313773.326492 │
│ KalimantanTengah ┆ 153711.086199 │
│ KalimantanBarat  ┆ 146869.654239 │
│ KalimantanTimur  ┆ 125829.729561 │
│ PapuaBarat       ┆ 98500.307117  │
│ Riau             ┆ 89874.498184  │
│ SumateraSelatan  ┆ 86986.344561  │
│ SumateraUtara    ┆ 72229.064051  │
│ KalimantanUtara  ┆ 70114.152441  │
│ SulawesiTengah   ┆ 61156.202508  │
└──────────────────┴───────────────┘
Tersimpan: /workspace/data/external/luas_provinsi.csv


## PENJELASAN
1. Dalam cell ini fokus penggabungan untuk kolom Nama provinsi dengan luasnya

In [18]:
joined_pl = pl.from_pandas(joined.drop(columns=["geometry", "index_right"], errors="ignore"))

data_pl = (
    joined_pl
    .with_columns(pl.col("acq_date").cast(pl.Utf8).str.to_date("%Y-%m-%d").alias("tanggal"))
    .with_columns(pl.col("tanggal").dt.month_start().alias("bulan"))
)

kalender_bulan = (
    pl.DataFrame({
        "bulan": pl.date_range(
            data_pl["bulan"].min(), data_pl["bulan"].max(), interval="1mo", eager=True
        )
    })
    .with_columns(pl.col("bulan").dt.month_end().dt.day().alias("hari_dalam_bulan"))
)
cakupan = (
    kalender_bulan.join(
        data_pl.group_by("bulan").agg(pl.col("tanggal").n_unique().alias("hari_terisi")),
        on="bulan",
        how="left",
    )
    .with_columns(pl.col("hari_terisi").fill_null(0))
    .sort("bulan")
)
bulan_tidak_lengkap = cakupan.filter(pl.col("hari_terisi") < pl.col("hari_dalam_bulan"))
bulan_kosong = cakupan.filter(pl.col("hari_terisi") == 0)  
print(f"Rentang tanggal: {data_pl['tanggal'].min()} s/d {data_pl['tanggal'].max()}")
print(f"Bulan dengan cakupan hari tidak penuh: {bulan_tidak_lengkap.height} dari {cakupan.height}")
print(f"Di antaranya bulan yang sama sekali kosong: {bulan_kosong.height}") 
print(bulan_tidak_lengkap)

semua_hari = pl.DataFrame({
    "tanggal": pl.date_range(
        data_pl["tanggal"].min(), data_pl["tanggal"].max(), interval="1d", eager=True
    )
})
hari_kosong = (
    semua_hari.join(data_pl.select("tanggal").unique(), on="tanggal", how="anti")
    .sort("tanggal")
)
print(
    f"Hari tanpa satu pun deteksi: {hari_kosong.height} dari {semua_hari.height} hari "
    f"({hari_kosong.height / semua_hari.height * 100:.2f}%)"
)
hari_kosong

Rentang tanggal: 2016-01-01 s/d 2026-06-30
Bulan dengan cakupan hari tidak penuh: 14 dari 126
Di antaranya bulan yang sama sekali kosong: 1
shape: (14, 3)
┌────────────┬──────────────────┬─────────────┐
│ bulan      ┆ hari_dalam_bulan ┆ hari_terisi │
│ ---        ┆ ---              ┆ ---         │
│ date       ┆ i8               ┆ u32         │
╞════════════╪══════════════════╪═════════════╡
│ 2021-02-01 ┆ 28               ┆ 27          │
│ 2022-07-01 ┆ 31               ┆ 26          │
│ 2022-08-01 ┆ 31               ┆ 21          │
│ 2023-07-01 ┆ 31               ┆ 29          │
│ 2024-05-01 ┆ 31               ┆ 29          │
│ 2024-06-01 ┆ 30               ┆ 27          │
│ 2024-07-01 ┆ 31               ┆ 20          │
│ 2024-09-01 ┆ 30               ┆ 29          │
│ 2024-11-01 ┆ 30               ┆ 27          │
│ 2025-11-01 ┆ 30               ┆ 29          │
│ 2026-03-01 ┆ 31               ┆ 28          │
│ 2026-04-01 ┆ 30               ┆ 27          │
│ 2026-05-01 ┆ 31            

tanggal
date
2021-02-01
2022-07-27
2022-07-28
2022-07-29
2022-07-30
2022-07-31
2022-08-01
2022-08-02
2022-08-03


## PENJELASAN 
1. Cell ini fokus dalam penggabungan hal bulan/tahun, lalu total hari dalam satu bulan, setelah dihitung berapa hari memiliki riwayat deteksi apinya.
2. Tabel tanggalan tersebut merupakan tanggal yang memiliki riwayat tidak melakukan deteksi atau tidak mendeteksi adanya api.

In [19]:
mentah_per_bulan = (
    lf.select(pl.col("acq_date").cast(pl.Utf8).str.slice(0, 7).alias("tahun_bulan"))
    .group_by("tahun_bulan")
    .agg(pl.len().alias("baris_mentah"))
    .collect()
)
cek_bulan_kosong = (
    bulan_kosong.with_columns(pl.col("bulan").dt.strftime("%Y-%m").alias("tahun_bulan"))
    .join(mentah_per_bulan, on="tahun_bulan", how="left")
    .select("tahun_bulan", "baris_mentah")
)
print("baris_mentah = null berarti bulan itu tidak ada sama sekali di file mentah")
cek_bulan_kosong

baris_mentah = null berarti bulan itu tidak ada sama sekali di file mentah


tahun_bulan,baris_mentah
str,u32
"""2026-05""",null


## PENJELASAN
1. Cell ini fokus hanya pengecekan apakah null ini ada di dataset mentah. Hasilnya benar bahwa null tidak ada di file mentah
2. Lalu ditunjukkan bulan yang memiliki null yaitu bulan 5 Atau Mei tahun 2026, yang dimana hasil pendeteksiannya belum dikeluarkan resmi.

In [ ]:
deteksi = (
    data_pl
    .group_by([PROV, "bulan"])
    .agg(
        pl.len().alias("jumlah_deteksi"),
        pl.col("frp").sum().alias("frp_total"),
        pl.col("frp").max().alias("frp_max"),
    )
)

semua_bulan = cakupan.select("bulan", "hari_dalam_bulan", "hari_terisi") 
grid = luas_provinsi.select(PROV).join(semua_bulan, how="cross")

assert deteksi.join(luas_provinsi, on=PROV, how="anti").height == 0, "Ada nama provinsi tanpa data luas"

agg_normalized = (
    grid.join(deteksi, on=[PROV, "bulan"], how="left")
    .with_columns(pl.col(["jumlah_deteksi", "frp_total", "frp_max"]).fill_null(0))
    
    .with_columns([
        pl.when(pl.col("hari_terisi") > 0).then(pl.col(c)).otherwise(None).alias(c)
        for c in ["jumlah_deteksi", "frp_total", "frp_max"]
    ])
    .with_columns(pl.col("bulan").dt.strftime("%Y-%m").alias("tahun_bulan"))
    .join(luas_provinsi, on=PROV, how="left")
    .with_columns(
        (pl.col("jumlah_deteksi") / pl.col("luas_km2") * 1000).alias("deteksi_per_1000km2")
    )
    .with_columns((pl.col("hari_terisi") == pl.col("hari_dalam_bulan")).alias("bulan_lengkap"))
    .select([PROV, "tahun_bulan", "jumlah_deteksi", "frp_total", "frp_max",
             "luas_km2", "deteksi_per_1000km2",
             "hari_terisi", "hari_dalam_bulan", "bulan_lengkap"])
    .sort([PROV, "tahun_bulan"])
)

n_zero = agg_normalized.filter(pl.col("jumlah_deteksi") == 0).height
print(f"Baris provinsi-bulan: {agg_normalized.height:,} ({luas_provinsi.height} provinsi x {semua_bulan.height} bulan)")
print(f"Di antaranya bulan ber-data tanpa deteksi (diisi 0): {n_zero:,}") 
print(f"Baris provinsi-bulan tanpa data sama sekali (null): {agg_normalized.filter(pl.col('hari_terisi') == 0).height:,}")  # [TAMBAH]
print(f"Bulan dengan flag bulan_lengkap = False: {semua_bulan.filter(pl.col('hari_terisi') < pl.col('hari_dalam_bulan')).height}")
audit.append(("total deteksi pada tabel agregat", int(agg_normalized["jumlah_deteksi"].sum())))
agg_normalized.head(10)

Baris provinsi-bulan: 4,284 (34 provinsi x 126 bulan)
Di antaranya bulan ber-data tanpa deteksi (diisi 0): 158
Baris provinsi-bulan tanpa data sama sekali (null): 34
Bulan dengan flag bulan_lengkap = False: 14


NAME_1,tahun_bulan,jumlah_deteksi,frp_total,frp_max,luas_km2,deteksi_per_1000km2,hari_terisi,hari_dalam_bulan,bulan_lengkap
str,str,u32,f64,f64,f64,f64,u32,i8,bool
"""Aceh""","""2016-01""",205,1303.06,50.37,56836.400597,3.606843,31,31,true
"""Aceh""","""2016-02""",442,3129.25,47.26,56836.400597,7.776706,29,29,true
"""Aceh""","""2016-03""",849,6361.36,171.37,56836.400597,14.93761,31,31,true
"""Aceh""","""2016-04""",502,4226.26,171.46,56836.400597,8.832368,30,30,true
"""Aceh""","""2016-05""",119,772.61,21.27,56836.400597,2.093729,31,31,true
"""Aceh""","""2016-06""",429,2521.51,47.27,56836.400597,7.54798,30,30,true
"""Aceh""","""2016-07""",571,4310.98,88.74,56836.400597,10.046379,31,31,true
"""Aceh""","""2016-08""",503,3004.29,71.0,56836.400597,8.849962,31,31,true
"""Aceh""","""2016-09""",410,2754.32,60.44,56836.400597,7.213687,30,30,true


## PENJELASAN ##
1. Luas provinsi dihitung dari polygon GADM yang diproyeksikan ke EPSG:6933 (equal-area) lalu dikonversi m² → km². Angka bisa berbeda dari BPS karena perbedaan presisi batas wilayah. Hasil disimpan ke `luas_provinsi.csv`.
2. Data dikelompokkan per provinsi dan bulan, dihitung `jumlah_deteksi`, serta `frp_total` dan `frp_max` sebagai ukuran intensitas.
3. Tabel dilengkapi grid provinsi × bulan. Bulan tanpa deteksi bernilai 0, bukan baris yang hilang. Untuk `frp_max`, nilai 0 berarti tidak ada deteksi pada bulan itu.
4. `deteksi_per_1000km2 = jumlah_deteksi / luas × 1000` agar provinsi besar tidak otomatis tampak lebih rawan hanya karena wilayahnya luas.

## 2.i Validasi

In [21]:
aturan_titik = {
    "latitude di luar rentang Indonesia": (pl.col("latitude") < -12) | (pl.col("latitude") > 7),
    "longitude di luar rentang Indonesia": (pl.col("longitude") < 94) | (pl.col("longitude") > 142),
    "frp negatif": pl.col("frp") < 0,
    "scan/track <= 0": (pl.col("scan") <= 0) | (pl.col("track") <= 0),
    "confidence bukan l/n/h": ~pl.col("confidence").is_in(["l", "n", "h"]),
    "type bukan 0-3": ~pl.col("type").is_in([0, 1, 2, 3]),
}
hasil_titik = lf_filtered.select(
    [expr.sum().alias(nama) for nama, expr in aturan_titik.items()]
).collect()
validasi_titik = hasil_titik.transpose(
    include_header=True, header_name="aturan", column_names=["pelanggaran"]
)
print(validasi_titik)
assert validasi_titik["pelanggaran"].sum() == 0, "Ada pelanggaran aturan level titik"

shape: (6, 2)
┌─────────────────────────────────┬─────────────┐
│ aturan                          ┆ pelanggaran │
│ ---                             ┆ ---         │
│ str                             ┆ u32         │
╞═════════════════════════════════╪═════════════╡
│ latitude di luar rentang Indon… ┆ 0           │
│ longitude di luar rentang Indo… ┆ 0           │
│ frp negatif                     ┆ 0           │
│ scan/track <= 0                 ┆ 0           │
│ confidence bukan l/n/h          ┆ 0           │
│ type bukan 0-3                  ┆ 0           │
└─────────────────────────────────┴─────────────┘


## PENJELASAN
1. Pada cell ini fokus pada cross check lanjutan apakah ada nilai yang tidak sesuai titik titik tertentu agar data semakin mengerucut sesuai dengan kebutuhan proses analisis lanjut.

In [22]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
agg_normalized.write_parquet(OUTPUT_PATH)
print(f"Tersimpan: {OUTPUT_PATH}")

PARQUET = OUTPUT_PATH.as_posix()
validasi_agregat = con.execute(f"""
    SELECT 'kunci provinsi-bulan duplikat' AS aturan, COUNT(*) AS pelanggaran FROM (
        SELECT "{PROV}", tahun_bulan FROM '{PARQUET}' GROUP BY 1, 2 HAVING COUNT(*) > 1)
    UNION ALL SELECT 'provinsi null', COUNT(*) FROM '{PARQUET}' WHERE "{PROV}" IS NULL
    UNION ALL SELECT 'jumlah_deteksi < 0', COUNT(*) FROM '{PARQUET}' WHERE jumlah_deteksi < 0
    UNION ALL SELECT 'luas_km2 null atau <= 0', COUNT(*) FROM '{PARQUET}'
        WHERE luas_km2 IS NULL OR luas_km2 <= 0
    -- [UBAH] null hanya boleh pada bulan tanpa data
    UNION ALL SELECT 'deteksi_per_1000km2 null (bulan ber-data) atau < 0', COUNT(*) FROM '{PARQUET}'
        WHERE (deteksi_per_1000km2 IS NULL AND hari_terisi > 0) OR deteksi_per_1000km2 < 0
    UNION ALL SELECT 'format tahun_bulan tidak valid', COUNT(*) FROM '{PARQUET}'
        WHERE NOT regexp_matches(tahun_bulan, '^[0-9]{{4}}-(0[1-9]|1[0-2])$')
    -- [UBAH] batas bawah hari_terisi dari 1 menjadi 0
    UNION ALL SELECT 'hari_terisi di luar 0..hari_dalam_bulan', COUNT(*) FROM '{PARQUET}'
        WHERE hari_terisi < 0 OR hari_terisi > hari_dalam_bulan
    -- [TAMBAH] dua aturan konsistensi null
    UNION ALL SELECT 'bulan tanpa data tetapi jumlah_deteksi tidak null', COUNT(*) FROM '{PARQUET}'
        WHERE hari_terisi = 0 AND jumlah_deteksi IS NOT NULL
    UNION ALL SELECT 'bulan ber-data tetapi jumlah_deteksi null', COUNT(*) FROM '{PARQUET}'
        WHERE hari_terisi > 0 AND jumlah_deteksi IS NULL
    UNION ALL SELECT 'bulan_lengkap tidak konsisten dengan hari_terisi', COUNT(*) FROM '{PARQUET}'
        WHERE bulan_lengkap <> (hari_terisi = hari_dalam_bulan)
""").pl()
print(validasi_agregat)
assert validasi_agregat["pelanggaran"].sum() == 0, "Ada pelanggaran aturan pada tabel agregat"

Tersimpan: /workspace/data/processed/hotspot_provinsi_bulanan.parquet
shape: (10, 2)
┌─────────────────────────────────┬─────────────┐
│ aturan                          ┆ pelanggaran │
│ ---                             ┆ ---         │
│ str                             ┆ i64         │
╞═════════════════════════════════╪═════════════╡
│ kunci provinsi-bulan duplikat   ┆ 0           │
│ provinsi null                   ┆ 0           │
│ jumlah_deteksi < 0              ┆ 0           │
│ luas_km2 null atau <= 0         ┆ 0           │
│ deteksi_per_1000km2 null (bula… ┆ 0           │
│ format tahun_bulan tidak valid  ┆ 0           │
│ hari_terisi di luar 0..hari_da… ┆ 0           │
│ bulan tanpa data tetapi jumlah… ┆ 0           │
│ bulan ber-data tetapi jumlah_d… ┆ 0           │
│ bulan_lengkap tidak konsisten … ┆ 0           │
└─────────────────────────────────┴─────────────┘


## PENJELASAN
1. Pada cell tersebut menghasilkan Tabel yang disimpan ke hotspot_provinsi_bulanan.parquet, lalu divalidasi dengan DuckDB memakai 10 aturan struktur: kunci provinsi-bulan unik, tidak ada provinsi null, jumlah_deteksi tidak negatif, luas > 0, deteksi_per_1000km2 valid, format tahun_bulan valid, hari_terisi dalam rentang 0 sampai hari_dalam_bulan, serta tiga aturan konsistensi antara null, hari_terisi, dan flag bulan_lengkap.
2. hasil validasi: 0 pelanggaran.


In [23]:
total_duckdb = con.execute(f"SELECT SUM(jumlah_deteksi) FROM '{PARQUET}'").fetchone()[0]
total_polars = int(agg_normalized["jumlah_deteksi"].sum())
total_sumber = len(joined)

print(f"Total DuckDB (parquet)      : {total_duckdb:,}")
print(f"Total Polars (agregat)      : {total_polars:,}")
print(f"Total titik sebelum agregasi: {total_sumber:,}")
assert total_duckdb == total_polars == total_sumber, "Total deteksi tidak konsisten antar tahap"

n_expected = luas_provinsi.height * semua_bulan.height
n_rows = con.execute(f"SELECT COUNT(*) FROM '{PARQUET}'").fetchone()[0]
assert n_rows == n_expected, f"Baris {n_rows} != provinsi x bulan {n_expected}"
print("Validasi lolos: parquet konsisten dengan agregasi Polars dan dengan jumlah titik sumber.")

Total DuckDB (parquet)      : 1,314,482
Total Polars (agregat)      : 1,314,482
Total titik sebelum agregasi: 1,314,482
Validasi lolos: parquet konsisten dengan agregasi Polars dan dengan jumlah titik sumber.


## PENJELASAN
1. Cell ini fokus terhadap hasil akhir perhitungan baris total DuckDB, Polars, dan sebelum di agregasi.

In [26]:
pl.Config.set_fmt_str_lengths(60) 
rekonsiliasi = pl.DataFrame(
    list(dict(audit).items()), schema=["tahap", "baris"], orient="row"
).with_columns(
    pl.col("baris").diff().alias("selisih_dari_tahap_sebelumnya")
)
rekonsiliasi

tahap,baris,selisih_dari_tahap_sebelumnya
str,i64,i64
"""mentah""",1535566,null
"""setelah drop null kolom wajib""",1535566,0
"""setelah deduplikasi""",1535566,0
"""setelah filter confidence & type""",1317362,-218204
"""setelah spatial join (tanpa duplikasi)""",1317362,0
"""setelah drop titik tanpa provinsi""",1314482,-2880
"""total deteksi pada tabel agregat""",1314482,0


## PENJELASAN
1. Cell ini fokus untuk mengetahui track total jumlah baris tiap tahapan dalam proses cleaning dengan hasil akhirnya yaitu 1,314,482 baris dengan selisih dari total data mentah sebelum dibersihkan sebanyak 221,084

## 2.j Lokasi file keluaran

In [27]:
for path in sorted((PROJECT_ROOT / "data").rglob("*")):
    if path.is_file():
        print(f"{path.relative_to(PROJECT_ROOT)}  ({path.stat().st_size / 1024:.1f} KB)")

data/README.md  (1.0 KB)
data/external/indonesia_provinces.geojson  (2894.0 KB)
data/external/luas_provinsi.csv  (1.0 KB)
data/processed/hotspot_provinsi_bulanan.parquet  (59.7 KB)
data/raw/fire_archive_SV-C2_812847.csv  (121591.9 KB)
